<a href="https://colab.research.google.com/github/ekonjmrivas-devops/llm_engineering/blob/mis-ejercicios/week7/Semana_7_D%C3%ADa_3_ENTRENAMIENTO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predecir precios de productos

## Día 3: Entrenamiento!


In [ ]:
# pip installs

!pip install -q datasets==2.21.0 requests torch peft bitsandbytes transformers trl accelerate sentencepiece wandb matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 11.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
!pip install fsspec==2025.3.0 gcsfs==2025.3.0

In [ ]:
# imports

import os
# Sistema operativo — acceder a variables de entorno, rutas de archivos

import re
# Expresiones regulares — búsqueda y manipulación de patrones en texto

import math
# Funciones matemáticas básicas (sqrt, log, etc.) — útiles en cálculos de métricas

from tqdm import tqdm
# Barra de progreso visual para loops — muestra avance durante iteraciones largas

from google.colab import userdata
# Acceso a secretos guardados en Google Colab (ej. tokens de API) de forma segura

from huggingface_hub import login
# Autenticarse en Hugging Face Hub con token — permite descargar modelos y subir resultados

import torch
# Framework de deep learning de PyTorch — tensores, GPU computing, autograd

import transformers
# Librería de Hugging Face con modelos pre-entrenados y utilidades de NLP

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
# Funciones específicas de transformers:
#   - AutoModelForCausalLM: carga automáticamente modelos generativos (ej. Llama 3.1)
#   - AutoTokenizer: carga automáticamente el tokenizador correspondiente al modelo
#   - TrainingArguments: define parámetros de entrenamiento (learning rate, batch size, épocas)
#   - set_seed: fija semilla aleatoria para reproducibilidad
#   - BitsAndBytesConfig: configuración de cuantización (4-bit, 8-bit)

from datasets import load_dataset, Dataset, DatasetDict
# Librería de Hugging Face para gestión de datasets:
#   - load_dataset: descarga/carga un dataset desde Hugging Face Hub o local
#   - Dataset: estructura de datos individual (una colección de ejemplos)
#   - DatasetDict: diccionario de Datasets (ej. separar train/validation/test)

import wandb
# Weights & Biases — plataforma de seguimiento y visualización de experimentos de entrenamiento (pérdida, métricas, gráficos en tiempo real)

from peft import LoraConfig
# PEFT (Parameter-Efficient Fine-Tuning) de Hugging Face:
#   - LoraConfig: define la configuración de LoRA (r, alpha, target_modules, dropout)

from trl import SFTTrainer, SFTConfig
# TRL (Transformer Reinforcement Learning) de Hugging Face, para fine-tuning supervisado:
#   - SFTTrainer: entrenador especializado en Supervised Fine-Tuning (SFT), integra LoRA/QLoRA automáticamente
#   - SFTConfig: configuración específica del entrenamiento SFT (extiende TrainingArguments)

from datetime import datetime
# Manejo de fechas y tiempos — para timestamps (ej. nombrar checkpoints con fecha/hora)

import matplotlib.pyplot as plt
# Librería de visualización — para graficar curvas de pérdida, métricas de entrenamiento, etc.

In [4]:
# Constantes

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
PROJECT_NAME = "pricer-data"
HF_USER = "jmrivas" # tu nombre de HF va aquí!

# Data

DATASET_NAME = f"{HF_USER}/pricer-data"
# O simplemente usa el que he subido
# DATASET_NAME = "joanby/pricer-data"
MAX_SEQUENCE_LENGTH = 182

# Nombre de la ejecución para guardar el modelo en el Hub

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hiper parámetros de QLoRA

LORA_R = 8 #32
LORA_ALPHA = 16 #64
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1
QUANT_4_BIT = True

# Hiper parámetros para el Entrenamiento

EPOCHS = 3
BATCH_SIZE = 1 #2, 4, 16
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"

# Configuración de Admin

STEPS = 50
SAVE_STEPS = 5000
LOG_TO_WANDB = True

%matplotlib inline

In [ ]:
HUB_MODEL_NAME

'jmrivas/pricer-data-2026-08-23_17.12.01'

# Más sobre optimizadores

https://huggingface.co/docs/transformers/main/en/perf_train_gpu_one#optimizer-choice

El más común es Adam o AdamW (Adam con decaimiento de peso).
Adam logra una buena convergencia al almacenar el promedio móvil de los gradientes anteriores; sin embargo, agrega una huella de memoria adicional del orden de la cantidad de parámetros del modelo.

### Inicia sesión en HuggingFace y Weights & Biases

Si aún no tienes una cuenta de HuggingFace, visita https://huggingface.co para registrarte y crear un token.

Luego, selecciona los secretos para este cuaderno haciendo clic en el ícono de la llave a la izquierda y agrega un nuevo secreto llamado `HF_TOKEN` con el valor como tu token.

Repite esto para weightsandbiases en https://wandb.ai y agrega un secreto llamado `WANDB_API_KEY`

In [17]:
# Log in en HuggingFace

hf_token = userdata.get('HF_KEY_W')
login(hf_token, add_to_git_credential=True)

In [7]:
# Log in en Weights & Biases
wandb_api_key = userdata.get('WANDB_AI')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configuramos Weights & Biases para almacenar la info de nuestro proyecto
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [8]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
test = dataset['test']

Generating train split:   0%|          | 0/400000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [9]:
test[0]

{'text': "¿Cuánto cuesta esto redondeado al dólar más cercano?\n\nOEM AC Compressor w/A/C Repair Kit For Ford F150 F-150 V8 & Lincoln Mark LT 2007 2008 - BuyAutoParts NEW\nAs one of the world's largest automotive parts suppliers, our parts are trusted every day by mechanics and vehicle owners worldwide. This A/C Compressor and Components Kit is manufactured and tested to the strictest OE standards for unparalleled performance. Built for trouble-free ownership and 100% visually inspected and quality tested, this A/C Compressor and Components Kit is backed by our 100% satisfaction guarantee. Guaranteed Exact Fit for easy installation 100% BRAND NEW, premium ISO/TS 16949 quality - tested to meet or exceed OEM specifications Engineered for superior durability, backed by industry-leading unlimited-mileage warranty Included in this K\n\nPrice is $",
 'price': 374.41}

In [10]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

wandb: Currently logged in as: ekon-jmrivas (ekon-jmrivas-individual) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Ahora cargue el tokenizador y el modelo

El modelo está "cuantificado": estamos reduciendo la precisión a 4 bits.

In [11]:
# Elegimos la opción de cuantización adecuada

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

In [12]:
# Cargamos el Tokenizer y el Modelo

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Memory footprint: 5591.5 MB


# Recopilador de datos

Es importante que nos aseguremos durante el entrenamiento de que no estamos intentando entrenar al modelo para predecir la descripción de los productos, sino solo su precio.

Debemos decirle al entrenador que todo hasta "El precio es $" está ahí para dar contexto al modelo para predecir el siguiente token, pero no es necesario aprenderlo.

El entrenador debe enseñarle al modelo a predecir el token o tokens después de "El precio es $".

Hay una forma complicada de hacer esto configurando máscaras, pero afortunadamente HuggingFace proporciona una clase auxiliar súper simple que se encarga de esto por nosotros.

In [13]:
from trl import DataCollatorForCompletionOnlyLM
response_template = "Price is $"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

# Y AHORA

## Configuramos la configuración para el entrenamiento

Necesitamos crear 2 objetos:

Un objeto LoraConfig con nuestros hiperparámetros para LoRA

Un SFTConfig con nuestros parámetros generales de entrenamiento

In [23]:
# Primero, especifique los parámetros de configuración para LoRA

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

#A continuación, especifique los parámetros de configuración general para el entrenamiento.

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    gradient_checkpointing=True
)

# Y ahora, el entrenador de ajuste fino supervisado realizará el ajuste fino
# Dados estos 2 conjuntos de parámetros de configuración

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    tokenizer=tokenizer,
    args=train_parameters,
    data_collator=collator
)

/tmp/ipykernel_3522/1678459276.py:47: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  fine_tuning = SFTTrainer(
/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying chat template to train dataset:   0%|          | 0/400000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/400000 [00:00<?, ? examples/s]

In [24]:
# Verifica cuántos adaptadores tiene el modelo
print(fine_tuning.model.peft_config)

{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path='meta-llama/Meta-Llama-3.1-8B', revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'v_proj', 'k_proj', 'o_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)}


In [25]:
from huggingface_hub import HfApi
api = HfApi()
models = api.list_models(author="jmrivas")
for m in models:
    print(m.modelId)

jmrivas/pricer-data-2026-08-23_18.28.37


In [26]:
# Fine-tune!
fine_tuning.train()

# Subimos nuestro modelo ajustado a Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Almacenado en el hub: {PROJECT_RUN_NAME}")

wandb: WARNING Artifact "model-2026-08-23_18.28.37" already exists with the same content. No new version will be created.


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
if LOG_TO_WANDB:
  wandb.finish()